In [ ]:
# 🤖 Aria AI Agent — Azure OpenAI Personal Assistant

This notebook implements an AI coding agent for **Aria**, a personal assistant built on Azure OpenAI and Azure services.

The agent can:

- **Manage the Aria codebase** — file operations, health checks, status reporting
- **Provide setup assistance** — environment validation, dependency checks, provider detection
- **Troubleshoot issues** — cross-stack diagnostics, log analysis, test execution
- **Answer questions** — codebase Q&A powered by Azure OpenAI GPT-4

---

### Architecture

```
User Query
    │
    ▼
┌─────────────────────────────────────────────┐
│              Aria AI Agent                  │
│  ┌───────────┐   ┌──────────────────────┐  │
│  │  Planner  │──▶│  Tool Router         │  │
│  └───────────┘   │  - codebase_search   │  │
│                  │  - run_health_check   │  │
│                  │  - run_tests         │  │
│                  │  - validate_env      │  │
│                  │  - read_file         │  │
│                  │  - get_status        │  │
│                  └──────────────────────┘  │
│                           │                │
│                           ▼                │
│              Azure OpenAI GPT-4            │
└─────────────────────────────────────────────┘
```


## 1. Install & Import Dependencies


In [ ]:
%pip install openai azure-identity httpx rich python-dotenv --quiet


In [ ]:
"""Core imports for the Aria AI Agent."""
import os
import sys
import json
import subprocess
import textwrap
from pathlib import Path
from typing import Any

# Third-party
from dotenv import load_dotenv
from openai import AzureOpenAI
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel
from rich.table import Table
from rich import print as rprint

# ── Repo root detection ──────────────────────────────────────────────────────
REPO_ROOT = Path("/workspaces/Aria")
if not REPO_ROOT.exists():
    # Fallback: walk up from notebook location
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent:
        if (REPO_ROOT / ".github" / "copilot-instructions.md").exists():
            break
        REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

console = Console()
console.print(f"[bold green]✅ Imports OK[/bold green] | Repo root: [cyan]{REPO_ROOT}[/cyan]")


## 2. Configuration — Azure OpenAI & Environment

Set your credentials via environment variables or `local.settings.json`.  
Priority order matches the Aria provider detection chain.


In [ ]:
def load_local_settings(repo_root: Path) -> dict[str, str]:
    """Load Azure Function local.settings.json Values section."""
    settings_path = repo_root / "local.settings.json"
    if settings_path.exists():
        with open(settings_path) as f:
            data = json.load(f)
        return data.get("Values", {})
    return {}


# ── Load env + local.settings.json ──────────────────────────────────────────
load_dotenv(REPO_ROOT / ".env", override=False)
local_settings = load_local_settings(REPO_ROOT)
for k, v in local_settings.items():
    os.environ.setdefault(k, v)

# ── Azure OpenAI config ──────────────────────────────────────────────────────
AZURE_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
AZURE_API_KEY = os.environ.get("AZURE_OPENAI_API_KEY", "")
AZURE_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")
AZURE_API_VERSION = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")

# ── Fallback config table ────────────────────────────────────────────────────
table = Table(title="🔧 Agent Configuration", show_lines=True)
table.add_column("Variable", style="cyan")
table.add_column("Status", style="bold")
table.add_column("Value (masked)", style="dim")

def _mask(v: str) -> str:
    return f"{v[:6]}…{v[-4:]}" if len(v) > 12 else ("✓ set" if v else "")

rows = [
    ("AZURE_OPENAI_ENDPOINT", AZURE_ENDPOINT),
    ("AZURE_OPENAI_API_KEY", AZURE_API_KEY),
    ("AZURE_OPENAI_DEPLOYMENT", AZURE_DEPLOYMENT),
    ("AZURE_OPENAI_API_VERSION", AZURE_API_VERSION),
]
for name, val in rows:
    status = "[green]✓ set[/green]" if val else "[red]✗ missing[/red]"
    table.add_row(name, status, _mask(val) if "KEY" in name else val)

console.print(table)

AZURE_READY = bool(AZURE_ENDPOINT and AZURE_API_KEY and AZURE_DEPLOYMENT)
if not AZURE_READY:
    console.print(
        "[yellow]⚠ Azure OpenAI not fully configured — agent will use local fallback echo mode.[/yellow]"
    )


## 3. Agent Tools

Each tool maps to a real operation in the Aria repo: file reads, health checks, test runs, env validation, and status queries.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool implementations
# Each function returns a plain dict consumed by the agent tool router.
# ─────────────────────────────────────────────────────────────────────────────

def tool_read_file(path: str, max_lines: int = 120) -> dict:
    """Read a file relative to REPO_ROOT (or absolute). Returns first max_lines lines."""
    target = Path(path) if Path(path).is_absolute() else REPO_ROOT / path
    if not target.exists():
        return {"error": f"File not found: {target}"}
    try:
        lines = target.read_text(encoding="utf-8", errors="replace").splitlines()
        truncated = len(lines) > max_lines
        return {
            "path": str(target),
            "lines": lines[:max_lines],
            "total_lines": len(lines),
            "truncated": truncated,
        }
    except Exception as exc:
        return {"error": str(exc)}


def tool_codebase_search(query: str, extensions: list[str] | None = None, max_results: int = 15) -> dict:
    """Grep for a pattern across the repo. Returns matching file:line snippets."""
    extensions = extensions or [".py", ".ts", ".js", ".yaml", ".md"]
    include_args = [f"--include=*{ext}" for ext in extensions]
    cmd = ["grep", "-rn", "--color=never", query, str(REPO_ROOT)] + include_args
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
        hits = result.stdout.strip().splitlines()[:max_results]
        return {"query": query, "matches": hits, "count": len(hits)}
    except subprocess.TimeoutExpired:
        return {"error": "Search timed out"}
    except Exception as exc:
        return {"error": str(exc)}


def tool_run_health_check() -> dict:
    """Run scripts/fast_validate.py and return a structured summary."""
    script = REPO_ROOT / "scripts" / "fast_validate.py"
    if not script.exists():
        return {"error": "fast_validate.py not found — skipping"}
    try:
        result = subprocess.run(
            [sys.executable, str(script)],
            capture_output=True, text=True, timeout=60, cwd=str(REPO_ROOT),
        )
        return {
            "returncode": result.returncode,
            "stdout": result.stdout[-3000:],
            "stderr": result.stderr[-1000:],
            "ok": result.returncode == 0,
        }
    except subprocess.TimeoutExpired:
        return {"error": "Health check timed out after 60 s"}
    except Exception as exc:
        return {"error": str(exc)}


def tool_run_tests(suite: str = "unit", extra_args: str = "") -> dict:
    """Run the Aria test suite via scripts/test_runner.py."""
    script = REPO_ROOT / "scripts" / "test_runner.py"
    if not script.exists():
        # Fallback to direct pytest
        cmd = [sys.executable, "-m", "pytest", "tests/", "-q", "--tb=short", "-x"]
    else:
        cmd = [sys.executable, str(script), f"--{suite}"]
    if extra_args:
        cmd += extra_args.split()
    try:
        result = subprocess.run(
            cmd, capture_output=True, text=True, timeout=120, cwd=str(REPO_ROOT),
        )
        return {
            "suite": suite,
            "returncode": result.returncode,
            "stdout": result.stdout[-4000:],
            "stderr": result.stderr[-1000:],
            "passed": result.returncode == 0,
        }
    except subprocess.TimeoutExpired:
        return {"error": f"Tests timed out after 120 s (suite={suite})"}
    except Exception as exc:
        return {"error": str(exc)}


def tool_validate_env() -> dict:
    """Check key environment variables and Python package availability."""
    env_vars = {
        "AZURE_OPENAI_ENDPOINT": os.environ.get("AZURE_OPENAI_ENDPOINT", ""),
        "AZURE_OPENAI_API_KEY": bool(os.environ.get("AZURE_OPENAI_API_KEY")),
        "AZURE_OPENAI_DEPLOYMENT": os.environ.get("AZURE_OPENAI_DEPLOYMENT", ""),
        "OPENAI_API_KEY": bool(os.environ.get("OPENAI_API_KEY")),
        "QAI_DB_CONN": bool(os.environ.get("QAI_DB_CONN")),
        "QAI_ENABLE_COSMOS": os.environ.get("QAI_ENABLE_COSMOS", "false"),
        "APPLICATIONINSIGHTS_CONNECTION_STRING": bool(
            os.environ.get("APPLICATIONINSIGHTS_CONNECTION_STRING")
        ),
    }
    packages = {}
    for pkg in ["openai", "torch", "transformers", "peft", "qiskit", "pennylane", "azure.identity"]:
        try:
            __import__(pkg.replace(".", "_") if "." in pkg else pkg)
            packages[pkg] = "✓"
        except ImportError:
            packages[pkg] = "✗ missing"
    return {"env_vars": env_vars, "packages": packages}


def tool_get_status() -> dict:
    """Read autonomous training + orchestrator status files from data_out/."""
    data_out = REPO_ROOT / "data_out"
    status_files = {}
    if data_out.exists():
        for sf in data_out.rglob("status.json"):
            try:
                relative = str(sf.relative_to(data_out))
                status_files[relative] = json.loads(sf.read_text())
            except Exception:
                status_files[str(sf)] = {"error": "Could not parse"}
    training_log = data_out / "autonomous_training.log"
    last_log_lines: list[str] = []
    if training_log.exists():
        lines = training_log.read_text(errors="replace").splitlines()
        last_log_lines = lines[-20:]
    return {
        "status_files": status_files,
        "last_training_log_lines": last_log_lines,
        "data_out_exists": data_out.exists(),
    }


def tool_list_directory(path: str = ".", max_entries: int = 50) -> dict:
    """List files and directories at a repo path."""
    target = Path(path) if Path(path).is_absolute() else REPO_ROOT / path
    if not target.exists():
        return {"error": f"Path not found: {target}"}
    try:
        entries = sorted(target.iterdir(), key=lambda p: (p.is_file(), p.name))
        listing = [
            {"name": e.name, "type": "dir" if e.is_dir() else "file", "size": e.stat().st_size if e.is_file() else None}
            for e in entries[:max_entries]
        ]
        return {"path": str(target), "entries": listing, "truncated": len(list(target.iterdir())) > max_entries}
    except Exception as exc:
        return {"error": str(exc)}


console.print("[bold green]✅ Tool functions registered[/bold green]")


## 4. OpenAI Tool Schema

Define the JSON schemas used by the Azure OpenAI function-calling API so the model can invoke tools autonomously.


In [ ]:
# ── OpenAI function-calling tool schemas ──────────────────────────────────────
TOOLS: list[dict] = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": (
                "Read any file in the Aria repository. "
                "Use for inspecting source code, configs, logs, or docs."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "File path relative to repo root, e.g. 'function_app.py' or 'shared/chat_providers.py'.",
                    },
                    "max_lines": {
                        "type": "integer",
                        "description": "Maximum lines to return (default 120).",
                        "default": 120,
                    },
                },
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "codebase_search",
            "description": (
                "Search the Aria codebase for a pattern using grep. "
                "Returns matching file:line snippets. Useful for finding where a function, "
                "class, or config key is used."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Grep pattern to search for.",
                    },
                    "extensions": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "File extensions to search, e.g. ['.py', '.yaml']. Defaults to common source types.",
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum number of matching lines to return (default 15).",
                        "default": 15,
                    },
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_health_check",
            "description": (
                "Run scripts/fast_validate.py to validate the Aria environment: "
                "datasets, configs, provider connectivity, venvs, and dependencies."
            ),
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_tests",
            "description": (
                "Run the Aria test suite. "
                "Use suite='unit' for fast checks, 'integration' for full coverage."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "suite": {
                        "type": "string",
                        "enum": ["unit", "integration", "all"],
                        "description": "Which test suite to run.",
                        "default": "unit",
                    },
                    "extra_args": {
                        "type": "string",
                        "description": "Extra pytest arguments, e.g. '-k chat --tb=short'.",
                        "default": "",
                    },
                },
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "validate_env",
            "description": (
                "Check environment variables and installed Python packages. "
                "Returns which Azure/OpenAI keys are set and which ML packages are available."
            ),
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_status",
            "description": (
                "Read orchestrator status files from data_out/ and recent autonomous training logs. "
                "Use to check training progress, cycle completions, and job health."
            ),
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "list_directory",
            "description": (
                "List files and sub-directories at a given path in the repo. "
                "Useful for exploring project structure."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Relative or absolute directory path.",
                        "default": ".",
                    },
                    "max_entries": {
                        "type": "integer",
                        "description": "Maximum entries to return.",
                        "default": 50,
                    },
                },
                "required": [],
            },
        },
    },
]

# ── Tool dispatcher ────────────────────────────────────────────────────────────
TOOL_DISPATCH: dict[str, Any] = {
    "read_file": tool_read_file,
    "codebase_search": tool_codebase_search,
    "run_health_check": tool_run_health_check,
    "run_tests": tool_run_tests,
    "validate_env": tool_validate_env,
    "get_status": tool_get_status,
    "list_directory": tool_list_directory,
}

console.print(f"[bold green]✅ {len(TOOLS)} tools registered[/bold green]: {[t['function']['name'] for t in TOOLS]}")


## 5. Agent Core — Agentic Loop

The agent runs an autonomous tool-calling loop:

1. Sends user query + system context to Azure OpenAI
2. If the model calls a tool → dispatch → inject result → continue
3. Repeats until the model produces a final text response (no tool calls)
4. Falls back to local echo mode when Azure OpenAI is unavailable


In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""
    You are Aria's AI development assistant — an expert coding agent for the Aria platform.

    Aria is an interactive AI character platform with:
    - Azure Functions API layer (function_app.py)
    - Multi-provider chat backends (LM Studio, Azure OpenAI, OpenAI, LoRA, local)
    - Autonomous LoRA training pipeline (scripts/autonomous_training_orchestrator.py)
    - Quantum ML integration (ai-projects/quantum-ml/)
    - Shared Python infrastructure (shared/)
    - Interactive 3D character web UI (apps/aria/)

    Your capabilities:
    - Read and analyse source files using read_file
    - Search the codebase for patterns using codebase_search
    - Run health checks with run_health_check
    - Execute tests with run_tests
    - Validate environment setup with validate_env
    - Check training/orchestrator status with get_status
    - Explore directory structure with list_directory

    Guidelines:
    - Always verify facts by reading actual files before answering code questions
    - Use codebase_search to find where a function or pattern is defined/used
    - Run health checks proactively when the user asks about environment problems
    - Be concise and actionable — give commands the user can run directly
    - Respect the safety rules: never suggest modifying datasets/, always --dry-run orchestrators first
    - Provider detection order: LM Studio → Ollama → Azure OpenAI → OpenAI → local
""").strip()


def _dispatch_tool(name: str, args: dict) -> str:
    """Execute a tool by name and return its result as a JSON string."""
    fn = TOOL_DISPATCH.get(name)
    if fn is None:
        return json.dumps({"error": f"Unknown tool: {name}"})
    try:
        result = fn(**args)
        return json.dumps(result, default=str)
    except Exception as exc:
        return json.dumps({"error": f"Tool execution failed: {exc}"})


def _local_fallback_response(user_query: str) -> str:
    """Echo-mode response when Azure OpenAI is not configured."""
    return (
        f"**[Local Fallback Mode — Azure OpenAI not configured]**\n\n"
        f"You asked: *{user_query}*\n\n"
        "To enable full AI responses, set these environment variables:\n"
        "```\n"
        "AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/\n"
        "AZURE_OPENAI_API_KEY=your-key-here\n"
        "AZURE_OPENAI_DEPLOYMENT=gpt-4o\n"
        "AZURE_OPENAI_API_VERSION=2024-12-01-preview\n"
        "```\n"
        "Or add them to `local.settings.json` → `Values`."
    )


def run_agent(
    user_query: str,
    *,
    max_tool_rounds: int = 8,
    verbose: bool = True,
) -> str:
    """
    Run the Aria AI agent for a single user query.

    Args:
        user_query: Natural language question or task.
        max_tool_rounds: Maximum tool-call iterations before forcing a final answer.
        verbose: Whether to print tool call/result summaries to console.

    Returns:
        Final assistant text response as a string.
    """
    if not AZURE_READY:
        return _local_fallback_response(user_query)

    client = AzureOpenAI(
        azure_endpoint=AZURE_ENDPOINT,
        api_key=AZURE_API_KEY,
        api_version=AZURE_API_VERSION,
    )

    messages: list[dict] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for round_num in range(max_tool_rounds + 1):
        response = client.chat.completions.create(
            model=AZURE_DEPLOYMENT,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=0.2,
            max_tokens=2048,
        )
        choice = response.choices[0]

        # ── Final answer — no tool calls ────────────────────────────────────
        if choice.finish_reason == "stop" or not choice.message.tool_calls:
            return choice.message.content or ""

        # ── Tool calls requested ─────────────────────────────────────────────
        tool_calls = choice.message.tool_calls
        messages.append({"role": "assistant", "content": choice.message.content, "tool_calls": [
            {
                "id": tc.id,
                "type": "function",
                "function": {"name": tc.function.name, "arguments": tc.function.arguments},
            }
            for tc in tool_calls
        ]})

        for tc in tool_calls:
            fn_name = tc.function.name
            try:
                fn_args = json.loads(tc.function.arguments)
            except json.JSONDecodeError:
                fn_args = {}

            if verbose:
                console.print(f"  [dim]🔧 Tool call:[/dim] [cyan]{fn_name}[/cyan] {json.dumps(fn_args, default=str)[:120]}")

            tool_result = _dispatch_tool(fn_name, fn_args)

            if verbose:
                preview = tool_result[:200].replace("\n", " ")
                console.print(f"  [dim]   ↳ result:[/dim] {preview}…" if len(tool_result) > 200 else f"  [dim]   ↳ result:[/dim] {preview}")

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": tool_result,
            })

    # ── Safety: force a final answer after max rounds ──────────────────────
    messages.append({"role": "user", "content": "Please provide your final answer now based on the information gathered."})
    final = client.chat.completions.create(
        model=AZURE_DEPLOYMENT,
        messages=messages,
        temperature=0.2,
        max_tokens=1024,
    )
    return final.choices[0].message.content or "No response generated."


console.print("[bold green]✅ Agent core ready[/bold green] — call [cyan]run_agent('your question')[/cyan] to start")


## 6. Rich Display Helper

Renders the agent's Markdown responses in the notebook output with syntax highlighting.


In [ ]:
def ask(query: str, *, max_tool_rounds: int = 8, verbose: bool = True) -> None:
    """
    Convenience wrapper: run the agent and pretty-print the response.

    Usage:
        ask("How does the provider detection chain work?")
        ask("Run a health check", verbose=False)
    """
    console.print(Panel(f"[bold white]{query}[/bold white]", title="💬 Query", border_style="blue"))

    with console.status("[bold cyan]Agent thinking…[/bold cyan]", spinner="dots"):
        answer = run_agent(query, max_tool_rounds=max_tool_rounds, verbose=verbose)

    console.print(Panel(Markdown(answer), title="🤖 Aria Agent", border_style="green", padding=(1, 2)))


console.print("[bold green]✅ ask() helper ready[/bold green]")
console.print()
console.print("[dim]Quick-start examples:[/dim]")
console.print('  [cyan]ask("What does function_app.py expose?")[/cyan]')
console.print('  [cyan]ask("Run a health check on the environment")[/cyan]')
console.print('  [cyan]ask("Show the provider detection chain")[/cyan]')
console.print('  [cyan]ask("Are there any failing tests?")[/cyan]')


## 7. Multi-Turn Conversation Agent

For interactive, stateful conversations with memory across turns.


In [ ]:
class AriaConversation:
    """
    Stateful multi-turn conversation agent with persistent message history.

    Each call to .chat() appends to history, enabling contextual follow-ups.
    Call .reset() to start a fresh session.

    Example:
        conv = AriaConversation()
        conv.chat("What providers does Aria support?")
        conv.chat("Which one is selected by default?")  # refers to previous context
        conv.reset()
    """

    def __init__(self, max_tool_rounds: int = 8, verbose: bool = True):
        self.max_tool_rounds = max_tool_rounds
        self.verbose = verbose
        self._history: list[dict] = [{"role": "system", "content": SYSTEM_PROMPT}]

    def reset(self) -> None:
        """Clear conversation history (keeps system prompt)."""
        self._history = [{"role": "system", "content": SYSTEM_PROMPT}]
        console.print("[dim]🔄 Conversation reset.[/dim]")

    def chat(self, user_message: str) -> str:
        """Send a message and get a response, preserving full conversation history."""
        if not AZURE_READY:
            answer = _local_fallback_response(user_message)
            console.print(Panel(Markdown(answer), title="🤖 Aria Agent (fallback)", border_style="yellow"))
            return answer

        client = AzureOpenAI(
            azure_endpoint=AZURE_ENDPOINT,
            api_key=AZURE_API_KEY,
            api_version=AZURE_API_VERSION,
        )

        console.print(Panel(f"[bold white]{user_message}[/bold white]", title="💬 You", border_style="blue"))
        self._history.append({"role": "user", "content": user_message})

        for _ in range(self.max_tool_rounds + 1):
            response = client.chat.completions.create(
                model=AZURE_DEPLOYMENT,
                messages=self._history,
                tools=TOOLS,
                tool_choice="auto",
                temperature=0.2,
                max_tokens=2048,
            )
            choice = response.choices[0]

            # ── Final answer ──────────────────────────────────────────────────
            if choice.finish_reason == "stop" or not choice.message.tool_calls:
                answer = choice.message.content or ""
                self._history.append({"role": "assistant", "content": answer})
                console.print(Panel(Markdown(answer), title="🤖 Aria Agent", border_style="green", padding=(1, 2)))
                return answer

            # ── Tool calls ────────────────────────────────────────────────────
            tool_calls = choice.message.tool_calls
            self._history.append({"role": "assistant", "content": choice.message.content, "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments},
                }
                for tc in tool_calls
            ]})

            for tc in tool_calls:
                fn_name = tc.function.name
                try:
                    fn_args = json.loads(tc.function.arguments)
                except json.JSONDecodeError:
                    fn_args = {}

                if self.verbose:
                    console.print(f"  [dim]🔧 Tool:[/dim] [cyan]{fn_name}[/cyan] {json.dumps(fn_args)[:100]}")

                tool_result = _dispatch_tool(fn_name, fn_args)

                if self.verbose:
                    preview = tool_result[:160].replace("\n", " ")
                    console.print(f"  [dim]   ↳[/dim] {preview}{'…' if len(tool_result) > 160 else ''}")

                self._history.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": tool_result,
                })

        # Force final answer
        self._history.append({"role": "user", "content": "Summarise your findings now."})
        final = client.chat.completions.create(
            model=AZURE_DEPLOYMENT, messages=self._history, temperature=0.2, max_tokens=1024,
        )
        answer = final.choices[0].message.content or ""
        self._history.append({"role": "assistant", "content": answer})
        console.print(Panel(Markdown(answer), title="🤖 Aria Agent", border_style="green", padding=(1, 2)))
        return answer

    @property
    def turn_count(self) -> int:
        """Number of user turns so far."""
        return sum(1 for m in self._history if m["role"] == "user")


# Instantiate a default conversation session
conv = AriaConversation()
console.print("[bold green]✅ AriaConversation ready[/bold green] — use [cyan]conv.chat('…')[/cyan] for multi-turn sessions")
console.print("[dim]  conv.reset() to start a new session | conv.turn_count to see history depth[/dim]")


## 8. Example Queries

Run these cells to try the agent. Each demonstrates a different capability.


In [ ]:
# ── Example 1: Codebase Q&A ───────────────────────────────────────────────────
# ask("What does function_app.py expose and how are routes structured?")


In [ ]:
# ── Example 2: Environment health check ──────────────────────────────────────
# ask("Run a health check and tell me what's missing or broken")


In [ ]:
# ── Example 3: Provider detection chain ──────────────────────────────────────
# ask("Explain the provider detection chain and which provider will be selected in my environment")


In [ ]:
# ── Example 4: Run unit tests and report failures ────────────────────────────
# ask("Run unit tests and summarise any failures")


## 9. Multi-Turn Example

Use `conv.chat()` for contextual follow-up questions within the same session.


In [ ]:
# Multi-turn conversation — each call retains context from the previous one.
# conv.chat("What AI providers does Aria support?")
# conv.chat("Which one will be picked in this environment?")  # follows up on previous
# conv.chat("How do I add a new provider?")                   # continues the thread
# conv.reset()                                                # clear history when done


## 10. Quick Reference

| Function                      | Description                        |
| ----------------------------- | ---------------------------------- |
| `ask("…")`                    | Single-turn query with rich output |
| `ask("…", verbose=False)`     | Suppress tool call logs            |
| `ask("…", max_tool_rounds=4)` | Limit tool iterations              |
| `conv.chat("…")`              | Multi-turn with persistent history |
| `conv.reset()`                | Clear conversation history         |
| `conv.turn_count`             | How many turns so far              |
| `run_agent("…")`              | Raw agent — returns string         |

### Available Tools

| Tool               | What it does                               |
| ------------------ | ------------------------------------------ |
| `read_file`        | Read any repo file (source, configs, logs) |
| `codebase_search`  | Grep across the codebase                   |
| `run_health_check` | Run `scripts/fast_validate.py`             |
| `run_tests`        | Run unit / integration test suite          |
| `validate_env`     | Check env vars + installed packages        |
| `get_status`       | Read orchestrator status files             |
| `list_directory`   | Explore directory structure                |

### Safe Defaults

- Datasets in `datasets/` are **read-only** — the agent never modifies them
- Orchestrators are always run with `--dry-run` first
- No secrets are logged or returned
